In [ ]:
# Configure the dataset root. Set PARKINSON_DATA_DIR to override the local default.
from pathlib import Path
import os

DATA_DIR = Path(os.environ.get("PARKINSON_DATA_DIR", "data")).expanduser().resolve()
TRAIN_DIR = DATA_DIR / "train images"
TEST_DIR = DATA_DIR / "test images"
if not TRAIN_DIR.is_dir():
    raise FileNotFoundError(f"Training directory not found: {TRAIN_DIR}. Set PARKINSON_DATA_DIR to the dataset root.")
if not TEST_DIR.is_dir():
    raise FileNotFoundError(f"Test directory not found: {TEST_DIR}. Set PARKINSON_DATA_DIR to the dataset root.")


The dataset is not included in this repository. Configure `PARKINSON_DATA_DIR` (or place the dataset in `./data`) before running the notebook; see `README.md` for the expected directory layout.


In [ ]:
#Import Libraries
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np 
import pandas as pd
from glob import glob 
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
from itertools import chain
from datetime import datetime 
import statistics
from tqdm import tqdm
from keras.callbacks import EarlyStopping
import h5py
from sklearn.metrics import classification_report , confusion_matrix
import seaborn as sn
from tensorflow.keras.activations import sigmoid
from tensorflow.keras.layers import Dense,Flatten,Conv2D,Dropout,MaxPooling2D,GlobalAveragePooling2D
from tensorflow.keras.models import Model,Sequential
from tensorflow.keras.metrics import Accuracy,Precision,Recall,AUC,BinaryCrossentropy,FalsePositives,FalseNegatives
from tensorflow.keras.callbacks import CSVLogger,ModelCheckpoint,ReduceLROnPlateau,LearningRateScheduler
from tensorflow.keras.losses import binary_crossentropy
from tensorflow.keras.optimizers import SGD,Adam
from keras import backend as k
from tensorflow.keras import Sequential
import keras 
import matplotlib
print('Started')

In [ ]:
#Extact Dataset
path , dirs , files=next(os.walk(TRAIN_DIR / 'pd'))
file_count=len(files)
print('Parkinson : ',file_count)
path , dirs , files=next(os.walk(TRAIN_DIR / 'control'))
file_count=len(files)
print('Non-Parkinson : ',file_count)

In [ ]:
#Reading Images
image_size=(224,224)
batch_size=32

train_df=tf.keras.preprocessing.image_dataset_from_directory(      #split train_data to train & validation
    str(TRAIN_DIR),
    validation_split=0.40,
    subset="training",
    seed=1337,
    image_size=image_size,
    batch_size=batch_size
)

val_df=tf.keras.preprocessing.image_dataset_from_directory(
    str(TRAIN_DIR),
    validation_split=0.40,
    subset="validation",
    seed=1337,
    image_size=image_size,
    batch_size=batch_size
)

test_df=tf.keras.preprocessing.image_dataset_from_directory(
    str(TEST_DIR),
    image_size=image_size,
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
#Find output for testing
y_true=np.concatenate([y for x,y in test_df],axis=0)

In [ ]:
#Hyperparameters
IMG_IND=224
IMG_SHAPE=(IMG_IND,IMG_IND,3)
LOSS=BinaryCrossentropy() #Not un-weighted
IMG_SIZE=(224,224)
SEED=2
BATCH_SIZE=16

In [ ]:
#Models Function
def get_callback(model_name):
  callbacks=[]

  checkpoint=tf.keras.callbacks.ModelCheckpoint(filepath=f'model.{model_name}.h5',verbose=1,monitor='val_loss',mode='min')
  callbacks.append(checkpoint)
  anne=ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=5,verbose=2,min_lr=0.00000001,min_delta=0.00001,mode='auto')
  callbacks.append(anne)
  earlystop=tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=10)
  callbacks.append(earlystop)

  return callbacks

**VGG16 Model**

In [ ]:
from keras.optimizers import Optimizer
OPTIMIZER=Adam(learning_rate=0.001,beta_1=0.9,beta_2=0.999)

base_model=tf.keras.applications.VGG16(input_shape=IMG_SHAPE,include_top=False,weights='imagenet')
base_model.trainable=True       #Not-Freezing
x=base_model.output             
x=GlobalAveragePooling2D()(x)   #Another method to Flatten
predictions=Dense(1,activation='sigmoid',name='Final')(x)     #Add output & flatten(x) to Predictions
model=Model(inputs=base_model.input ,outputs=predictions)     #Concatenate all layers(input) & Predictions(output)
#model.load_weights(dire)
model.compile(loss='binary_crossentropy',optimizer=OPTIMIZER,metrics=['accuracy',keras.metrics.Precision(),keras.metrics.Recall(),keras.metrics.SpecificityAtSensitivity(0.5)])

In [ ]:
#Print my model
print(model.summary())

In [ ]:
#Fitting for dense model
callbacks=get_callback('vgg16')
history=model.fit(train_df,epochs=5,
                  validation_data=val_df,
                  callbacks=[callbacks])

In [ ]:
#Show Graphs
acc=history.history['accuracy']
val_acc=history.history['val_accuracy']
loss=history.history['loss']
val_loss=history.history['val_loss']
epochs=range(1,len(acc)+1)


#Train & Validation accuracy
plt.plot(epochs, acc, label='Training accuracy')
plt.plot(epochs,val_acc,label='Validation accuracy')
plt.title('Training and Validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.figure()


#Train & Validation loss
plt.plot(epochs,loss,label='Training loss')
plt.plot(epochs,val_loss,label='Validation loss')
plt.title('Training and Validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
#After we save the model we can load it
model=keras.models.load_model('model.vgg16.h5')

In [ ]:
results=model.evaluate(test_df,verbose=1)

In [ ]:
#Print classification report
y_pred=model.predict(test_df,verbose=1).round()
y_pred=y_pred.flatten()
target_names=['class non parkinson' , 'class parkinson']
print(classification_report(y_true, y_pred, target_names=target_names,digits=4))

In [ ]:
#Print confusion matrix
target_names=['Parkinson' , 'Non Parkinson']
report=classification_report(y_true, y_pred, target_names=target_names,digits=4)
print(report)

disp=confusion_matrix(y_true ,y_pred)
disp.astype('int')
pd.options.display.float_format='{:.5f}'.format
df_cm=pd.DataFrame(disp , target_names , target_names)
#plt.figure(figsize=(10,7))
fig,ax=plt.subplots(figsize=(4,4))
sn.set(font_scale=1.5)   #for lable size
sn.heatmap(df_cm , annot=True , annot_kws={"size":15},ax=ax,cmap="YlOrBr", fmt='g', cbar=False)
plt.ylabel('Actual',fontsize=20)
plt.xlabel('Predicted',fontsize=20)
plt.ioff()
plt.savefig("confusion_mat" ,bbox_inches='tight')
plt.show()

**DenseNet121 Model**

In [ ]:
from keras.optimizers import Optimizer
OPTIMIZER=Adam(learning_rate=0.001,beta_1=0.9,beta_2=0.999)

base_model=tf.keras.applications.DenseNet121(input_shape=IMG_SHAPE,include_top=False,weights='imagenet')
base_model.trainable=True       #Not-Freezing
x=base_model.output             
x=GlobalAveragePooling2D()(x)   #Another method to Flatten
predictions=Dense(1,activation='sigmoid',name='Final')(x)     #Add output & flatten(x) to Predictions
model=Model(inputs=base_model.input ,outputs=predictions)     #Concatenate all layers(input) & Predictions(output)
#model.load_weights(dire)
model.compile(loss='binary_crossentropy',optimizer=OPTIMIZER,metrics=['accuracy',keras.metrics.Precision(),keras.metrics.Recall(),keras.metrics.SpecificityAtSensitivity(0.5)])

In [ ]:
#Print my model
print(model.summary())

In [ ]:
#Fitting for dense model
callbacks=get_callback('dense121')
history=model.fit(train_df,epochs=5,
                  validation_data=val_df,
                  callbacks=[callbacks])

In [ ]:
#Show Graphs
acc=history.history['accuracy']
val_acc=history.history['val_accuracy']
loss=history.history['loss']
val_loss=history.history['val_loss']
epochs=range(1,len(acc)+1)


#Train & Validation accuracy
plt.plot(epochs, acc, label='Training accuracy')
plt.plot(epochs,val_acc,label='Validation accuracy')
plt.title('Training and Validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.figure()


#Train & Validation loss
plt.plot(epochs,loss,label='Training loss')
plt.plot(epochs,val_loss,label='Validation loss')
plt.title('Training and Validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
#After we save the model we can load it
model=keras.models.load_model('model.dense121.h5')

In [ ]:
results=model.evaluate(test_df,verbose=1)

In [ ]:
#Print classification report
y_pred=model.predict(test_df,verbose=1).round()
y_pred=y_pred.flatten()
target_names=['class non parkinson' , 'class parkinson']
print(classification_report(y_true, y_pred, target_names=target_names,digits=4))

In [ ]:
#Print confusion matrix
target_names=['Parkinson' , 'Non Parkinson']
report=classification_report(y_true, y_pred, target_names=target_names,digits=4)
print(report)

disp=confusion_matrix(y_true ,y_pred)
disp.astype('int')
pd.options.display.float_format='{:.5f}'.format
df_cm=pd.DataFrame(disp , target_names , target_names)
#plt.figure(figsize=(10,7))
fig,ax=plt.subplots(figsize=(4,4))
sn.set(font_scale=1.5)   #for lable size
sn.heatmap(df_cm , annot=True , annot_kws={"size":15},ax=ax,cmap="YlOrBr", fmt='g', cbar=False)
plt.ylabel('Actual',fontsize=20)
plt.xlabel('Predicted',fontsize=20)
plt.ioff()
plt.savefig("confusion_mat" ,bbox_inches='tight')
plt.show()

**Xception Model** 

In [ ]:
from keras.optimizers import Optimizer
OPTIMIZER=Adam(learning_rate=0.001,beta_1=0.9,beta_2=0.999)

base_model=tf.keras.applications.Xception(input_shape=IMG_SHAPE,include_top=False,weights='imagenet')
base_model.trainable=True       #Not-Freezing
x=base_model.output             
x=GlobalAveragePooling2D()(x)   #Another method to Flatten
predictions=Dense(1,activation='sigmoid',name='Final')(x)     #Add output & flatten(x) to Predictions
model=Model(inputs=base_model.input ,outputs=predictions)     #Concatenate all layers(input) & Predictions(output)
#model.load_weights(dire)
model.compile(loss='binary_crossentropy',optimizer=OPTIMIZER,metrics=['accuracy',keras.metrics.Precision(),keras.metrics.Recall(),keras.metrics.SpecificityAtSensitivity(0.5)])

In [ ]:
#Print my model
print(model.summary())

In [ ]:
#Fitting for xception model
callbacks=get_callback('xception')
history=model.fit(train_df,epochs=5,
                  validation_data=val_df,
                  callbacks=[callbacks])

In [ ]:
#Show Graphs
acc=history.history['accuracy']
val_acc=history.history['val_accuracy']
loss=history.history['loss']
val_loss=history.history['val_loss']
epochs=range(1,len(acc)+1)


#Train & Validation accuracy
plt.plot(epochs, acc, label='Training accuracy')
plt.plot(epochs,val_acc,label='Validation accuracy')
plt.title('Training and Validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.figure()


#Train & Validation loss
plt.plot(epochs,loss,label='Training loss')
plt.plot(epochs,val_loss,label='Validation loss')
plt.title('Training and Validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
#After we save the model we can load it
model=keras.models.load_model('model.xception.h5')

In [ ]:
results=model.evaluate(test_df,verbose=1)

In [ ]:
#Print classification report
y_pred=model.predict(test_df,verbose=1).round()
y_pred=y_pred.flatten()
target_names=['class non parkinson' , 'class parkinson']
print(classification_report(y_true, y_pred, target_names=target_names,digits=4))

In [ ]:
#Print confusion matrix
target_names=['Parkinson' , 'Non Parkinson']
report=classification_report(y_true, y_pred, target_names=target_names,digits=4)
print(report)

disp=confusion_matrix(y_true ,y_pred)
disp.astype('int')
pd.options.display.float_format='{:.5f}'.format
df_cm=pd.DataFrame(disp , target_names , target_names)
#plt.figure(figsize=(10,7))
fig,ax=plt.subplots(figsize=(4,4))
sn.set(font_scale=1.5)   #for lable size
sn.heatmap(df_cm , annot=True , annot_kws={"size":15},ax=ax,cmap="YlOrBr", fmt='g', cbar=False)
plt.ylabel('Actual',fontsize=20)
plt.xlabel('Predicted',fontsize=20)
plt.ioff()
plt.savefig("confusion_mat" ,bbox_inches='tight')
plt.show()

**MobileNetv2 Model**

In [ ]:
from keras.optimizers import Optimizer
OPTIMIZER=Adam(learning_rate=0.001,beta_1=0.9,beta_2=0.999)

base_model=tf.keras.applications.MobileNetV2(input_shape=IMG_SHAPE,include_top=False,weights='imagenet')
base_model.trainable=True       #Not-Freezing
x=base_model.output             
x=GlobalAveragePooling2D()(x)   #Another method to Flatten
predictions=Dense(1,activation='sigmoid',name='Final')(x)     #Add output & flatten(x) to Predictions
model=Model(inputs=base_model.input ,outputs=predictions)     #Concatenate all layers(input) & Predictions(output)
#model.load_weights(dire)
model.compile(loss='binary_crossentropy',optimizer=OPTIMIZER,metrics=['accuracy',keras.metrics.Precision(),keras.metrics.Recall(),keras.metrics.SpecificityAtSensitivity(0.5)])

In [ ]:
#Print my model
print(model.summary())

In [ ]:
#Fitting for mobilenet model
callbacks=get_callback('mobilenet')
history=model.fit(train_df,epochs=5,
                  validation_data=val_df,
                  callbacks=[callbacks])

In [ ]:
#Show Graphs
acc=history.history['accuracy']
val_acc=history.history['val_accuracy']
loss=history.history['loss']
val_loss=history.history['val_loss']
epochs=range(1,len(acc)+1)


#Train & Validation accuracy
plt.plot(epochs, acc, label='Training accuracy')
plt.plot(epochs,val_acc,label='Validation accuracy')
plt.title('Training and Validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.figure()


#Train & Validation loss
plt.plot(epochs,loss,label='Training loss')
plt.plot(epochs,val_loss,label='Validation loss')
plt.title('Training and Validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
#After we save the model we can load it
model=keras.models.load_model('model.mobilenet.h5')

In [ ]:
results=model.evaluate(test_df,verbose=1)

In [ ]:
#Print classification report
y_pred=model.predict(test_df,verbose=1).round()
y_pred=y_pred.flatten()
target_names=['class non parkinson' , 'class parkinson']
print(classification_report(y_true, y_pred, target_names=target_names,digits=4))

In [ ]:
#Print confusion matrix
target_names=['Parkinson' , 'Non Parkinson']
report=classification_report(y_true, y_pred, target_names=target_names,digits=4)
print(report)

disp=confusion_matrix(y_true ,y_pred)
disp.astype('int')
pd.options.display.float_format='{:.5f}'.format
df_cm=pd.DataFrame(disp , target_names , target_names)
#plt.figure(figsize=(10,7))
fig,ax=plt.subplots(figsize=(4,4))
sn.set(font_scale=1.5)   #for lable size
sn.heatmap(df_cm , annot=True , annot_kws={"size":15},ax=ax,cmap="YlOrBr", fmt='g', cbar=False)
plt.ylabel('Actual',fontsize=20)
plt.xlabel('Predicted',fontsize=20)
plt.ioff()
plt.savefig("confusion_mat" ,bbox_inches='tight')
plt.show()

**ResNet50 Model**

In [ ]:
from keras.optimizers import Optimizer
OPTIMIZER=Adam(learning_rate=0.001,beta_1=0.9,beta_2=0.999)

base_model=tf.keras.applications.ResNet50(input_shape=IMG_SHAPE,include_top=False,weights='imagenet')
base_model.trainable=True       #Not-Freezing
x=base_model.output             
x=GlobalAveragePooling2D()(x)   #Another method to Flatten
predictions=Dense(1,activation='sigmoid',name='Final')(x)     #Add output & flatten(x) to Predictions
model=Model(inputs=base_model.input ,outputs=predictions)     #Concatenate all layers(input) & Predictions(output)
#model.load_weights(dire)
model.compile(loss='binary_crossentropy',optimizer=OPTIMIZER,metrics=['accuracy',keras.metrics.Precision(),keras.metrics.Recall(),keras.metrics.SpecificityAtSensitivity(0.5)])

In [ ]:
#Print my model
print(model.summary())

In [ ]:
#Fitting for resnet model
callbacks=get_callback('resnet')
history=model.fit(train_df,epochs=5,
                  validation_data=val_df,
                  callbacks=[callbacks])

In [ ]:
#Show Graphs
acc=history.history['accuracy']
val_acc=history.history['val_accuracy']
loss=history.history['loss']
val_loss=history.history['val_loss']
epochs=range(1,len(acc)+1)


#Train & Validation accuracy
plt.plot(epochs, acc, label='Training accuracy')
plt.plot(epochs,val_acc,label='Validation accuracy')
plt.title('Training and Validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.figure()


#Train & Validation loss
plt.plot(epochs,loss,label='Training loss')
plt.plot(epochs,val_loss,label='Validation loss')
plt.title('Training and Validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
#After we save the model we can load it
model=keras.models.load_model('model.resnet.h5')

In [ ]:
results=model.evaluate(test_df,verbose=1)

In [ ]:
#Print classification report
y_pred=model.predict(test_df,verbose=1).round()
y_pred=y_pred.flatten()
target_names=['class non parkinson' , 'class parkinson']
print(classification_report(y_true, y_pred, target_names=target_names,digits=4))

In [ ]:
#Print confusion matrix
target_names=['Parkinson' , 'Non Parkinson']
report=classification_report(y_true, y_pred, target_names=target_names,digits=4)
print(report)

disp=confusion_matrix(y_true ,y_pred)
disp.astype('int')
pd.options.display.float_format='{:.5f}'.format
df_cm=pd.DataFrame(disp , target_names , target_names)
#plt.figure(figsize=(10,7))
fig,ax=plt.subplots(figsize=(4,4))
sn.set(font_scale=1.5)   #for lable size
sn.heatmap(df_cm , annot=True , annot_kws={"size":15},ax=ax,cmap="YlOrBr", fmt='g', cbar=False)
plt.ylabel('Actual',fontsize=20)
plt.xlabel('Predicted',fontsize=20)
plt.ioff()
plt.savefig("confusion_mat" ,bbox_inches='tight')
plt.show()

**InceptionV3 Model**

In [ ]:
from keras.optimizers import Optimizer
OPTIMIZER=Adam(learning_rate=0.001,beta_1=0.9,beta_2=0.999)

base_model=tf.keras.applications.InceptionV3(input_shape=IMG_SHAPE,include_top=False,weights='imagenet')
base_model.trainable=True       #Not-Freezing
x=base_model.output             
x=GlobalAveragePooling2D()(x)   #Another method to Flatten
predictions=Dense(1,activation='sigmoid',name='Final')(x)     #Add output & flatten(x) to Predictions
model=Model(inputs=base_model.input ,outputs=predictions)     #Concatenate all layers(input) & Predictions(output)
#model.load_weights(dire)
model.compile(loss='binary_crossentropy',optimizer=OPTIMIZER,metrics=['accuracy',keras.metrics.Precision(),keras.metrics.Recall(),keras.metrics.SpecificityAtSensitivity(0.5)])

In [ ]:
#Print my model
print(model.summary())

In [ ]:
#Fitting for inception model
callbacks=get_callback('inception')
history=model.fit(train_df,epochs=5,
                  validation_data=val_df,
                  callbacks=[callbacks])

In [ ]:
#Show Graphs
acc=history.history['accuracy']
val_acc=history.history['val_accuracy']
loss=history.history['loss']
val_loss=history.history['val_loss']
epochs=range(1,len(acc)+1)


#Train & Validation accuracy
plt.plot(epochs, acc, label='Training accuracy')
plt.plot(epochs,val_acc,label='Validation accuracy')
plt.title('Training and Validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.figure()


#Train & Validation loss
plt.plot(epochs,loss,label='Training loss')
plt.plot(epochs,val_loss,label='Validation loss')
plt.title('Training and Validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
#After we save the model we can load it
model=keras.models.load_model('model.inception.h5')

In [ ]:
results=model.evaluate(test_df,verbose=1)

In [ ]:
#Print classification report
y_pred=model.predict(test_df,verbose=1).round()
y_pred=y_pred.flatten()
target_names=['class non parkinson' , 'class parkinson']
print(classification_report(y_true, y_pred, target_names=target_names,digits=4))

In [ ]:
#Print confusion matrix
target_names=['Parkinson' , 'Non Parkinson']
report=classification_report(y_true, y_pred, target_names=target_names,digits=4)
print(report)

disp=confusion_matrix(y_true ,y_pred)
disp.astype('int')
pd.options.display.float_format='{:.5f}'.format
df_cm=pd.DataFrame(disp , target_names , target_names)
#plt.figure(figsize=(10,7))
fig,ax=plt.subplots(figsize=(4,4))
sn.set(font_scale=1.5)   #for lable size
sn.heatmap(df_cm , annot=True , annot_kws={"size":15},ax=ax,cmap="YlOrBr", fmt='g', cbar=False)
plt.ylabel('Actual',fontsize=20)
plt.xlabel('Predicted',fontsize=20)
plt.ioff()
plt.savefig("confusion_mat" ,bbox_inches='tight')
plt.show()

**InceptionResNetV2 Model**

In [ ]:
from keras.optimizers import Optimizer
OPTIMIZER=Adam(learning_rate=0.001,beta_1=0.9,beta_2=0.999)

base_model=tf.keras.applications.InceptionResNetV2(input_shape=IMG_SHAPE,include_top=False,weights='imagenet')
base_model.trainable=True       #Not-Freezing
x=base_model.output             
x=GlobalAveragePooling2D()(x)   #Another method to Flatten
predictions=Dense(1,activation='sigmoid',name='Final')(x)     #Add output & flatten(x) to Predictions
model=Model(inputs=base_model.input ,outputs=predictions)     #Concatenate all layers(input) & Predictions(output)
#model.load_weights(dire)
model.compile(loss='binary_crossentropy',optimizer=OPTIMIZER,metrics=['accuracy',keras.metrics.Precision(),keras.metrics.Recall(),keras.metrics.SpecificityAtSensitivity(0.5)])

In [ ]:
#Print my model
print(model.summary())

In [ ]:
#Fitting for InceptionResNetV2 model
callbacks=get_callback('InceptionResNetV2')
history=model.fit(train_df,epochs=5,
                  validation_data=val_df,
                  callbacks=[callbacks])

In [ ]:
#Show Graphs
acc=history.history['accuracy']
val_acc=history.history['val_accuracy']
loss=history.history['loss']
val_loss=history.history['val_loss']
epochs=range(1,len(acc)+1)


#Train & Validation accuracy
plt.plot(epochs, acc, label='Training accuracy')
plt.plot(epochs,val_acc,label='Validation accuracy')
plt.title('Training and Validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.figure()


#Train & Validation loss
plt.plot(epochs,loss,label='Training loss')
plt.plot(epochs,val_loss,label='Validation loss')
plt.title('Training and Validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
#After we save the model we can load it
model=keras.models.load_model('model.InceptionResNetV2.h5') 

In [ ]:
results=model.evaluate(test_df,verbose=1)

In [ ]:
#Print classification report
y_pred=model.predict(test_df,verbose=1).round()
y_pred=y_pred.flatten()
target_names=['class non parkinson' , 'class parkinson']
print(classification_report(y_true, y_pred, target_names=target_names,digits=4))

In [ ]:
#Print confusion matrix
target_names=['Parkinson' , 'Non Parkinson']
report=classification_report(y_true, y_pred, target_names=target_names,digits=4)
print(report)

disp=confusion_matrix(y_true ,y_pred)
disp.astype('int')
pd.options.display.float_format='{:.5f}'.format
df_cm=pd.DataFrame(disp , target_names , target_names)
#plt.figure(figsize=(10,7))
fig,ax=plt.subplots(figsize=(4,4))
sn.set(font_scale=1.5)   #for lable size
sn.heatmap(df_cm , annot=True , annot_kws={"size":15},ax=ax,cmap="YlOrBr", fmt='g', cbar=False)
plt.ylabel('Actual',fontsize=20)
plt.xlabel('Predicted',fontsize=20)
plt.ioff()
plt.savefig("confusion_mat" ,bbox_inches='tight')
plt.show()

**DenseNet201 Model**

In [ ]:
from keras.optimizers import Optimizer
OPTIMIZER=Adam(learning_rate=0.001,beta_1=0.9,beta_2=0.999)

base_model=tf.keras.applications.DenseNet201(input_shape=IMG_SHAPE,include_top=False,weights='imagenet')
base_model.trainable=True       #Not-Freezing
x=base_model.output             
x=GlobalAveragePooling2D()(x)   #Another method to Flatten
predictions=Dense(1,activation='sigmoid',name='Final')(x)     #Add output & flatten(x) to Predictions
model=Model(inputs=base_model.input ,outputs=predictions)     #Concatenate all layers(input) & Predictions(output)
#model.load_weights(dire)
model.compile(loss='binary_crossentropy',optimizer=OPTIMIZER,metrics=['accuracy',keras.metrics.Precision(),keras.metrics.Recall(),keras.metrics.SpecificityAtSensitivity(0.5)])

In [ ]:
#Print my model
print(model.summary())

In [ ]:
#Fitting for DenseNet201 model
callbacks=get_callback('DenseNet201')
history=model.fit(train_df,epochs=5,
                  validation_data=val_df,
                  callbacks=[callbacks])

In [ ]:
#Show Graphs
acc=history.history['accuracy']
val_acc=history.history['val_accuracy']
loss=history.history['loss']
val_loss=history.history['val_loss']
epochs=range(1,len(acc)+1)


#Train & Validation accuracy
plt.plot(epochs, acc, label='Training accuracy')
plt.plot(epochs,val_acc,label='Validation accuracy')
plt.title('Training and Validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.figure()


#Train & Validation loss
plt.plot(epochs,loss,label='Training loss')
plt.plot(epochs,val_loss,label='Validation loss')
plt.title('Training and Validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
#After we save the model we can load it
model=keras.models.load_model('model.DenseNet201.h5') 

In [ ]:
results=model.evaluate(test_df,verbose=1)

In [ ]:
#Print classification report
y_pred=model.predict(test_df,verbose=1).round()
y_pred=y_pred.flatten()
target_names=['class non parkinson' , 'class parkinson']
print(classification_report(y_true, y_pred, target_names=target_names,digits=4))

In [ ]:
#Print confusion matrix
target_names=['Parkinson' , 'Non Parkinson']
report=classification_report(y_true, y_pred, target_names=target_names,digits=4)
print(report)

disp=confusion_matrix(y_true ,y_pred)
disp.astype('int')
pd.options.display.float_format='{:.5f}'.format
df_cm=pd.DataFrame(disp , target_names , target_names)
#plt.figure(figsize=(10,7))
fig,ax=plt.subplots(figsize=(4,4))
sn.set(font_scale=1.5)   #for lable size
sn.heatmap(df_cm , annot=True , annot_kws={"size":15},ax=ax,cmap="YlOrBr", fmt='g', cbar=False)
plt.ylabel('Actual',fontsize=20)
plt.xlabel('Predicted',fontsize=20)
plt.ioff()
plt.savefig("confusion_mat" ,bbox_inches='tight')
plt.show()